# Broad MC-Dropout Rate Sweep

This notebook scores the official 500K pool with broad attention, residual, and embedding dropout at `p in [0, 0.00001, 0.001, 0.01]`. The zero-dropout control uses `K=1`; stochastic configurations use `K=8`. All metadata and deterministic reference arrays are aligned through `score_index`.

Raw scoring artifacts are isolated by configuration and shard. The existing alignment-corrected `p=0.05` result is read only as an optional report reference.

## 0. Resource Assumptions

Use a Colab GPU runtime with Python 3.12 and PyTorch 2.10.x or 2.11.x. Require at least 12 GB local scratch and 10 GB free Drive space for the isolated raw score shards, indexes, logs, and reports. The benchmark prints a measured conservative ETA before production. Every long subprocess emits flushed progress or a 30-second heartbeat.

In [ ]:
# PYTHON CELL
import json, os, shutil, subprocess, sys
from pathlib import Path

subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['df', '-h', '/content'], check=True)
if shutil.disk_usage('/content').free < 12_000_000_000:
    raise RuntimeError('At least 12 GB of local disk is required for the full-pool raw token copy and analysis.')
print('Python:', sys.version)

## 1. Runtime and Google Drive

**One-time setup.** Mount Drive before any Drive path is referenced.

In [ ]:
# PYTHON CELL
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# PYTHON CELL
DRIVE = Path('/content/drive/MyDrive/color-filter-ablation')
DATA_DRIVE = DRIVE / 'data'
MODELS_DRIVE = DRIVE / 'assets' / 'raw' / 'models'
SOURCE_RESULTS = DRIVE / 'results'
STAGE_ROOT = SOURCE_RESULTS / 'dropout-uncertainty-broad-rate-sweep'
RAW_SCORE_DRIVE = STAGE_ROOT / 'raw_score_shards'
CONFIG_DRIVE = DRIVE / 'runtime_configs' / 'dropout-uncertainty-broad-rate-sweep'
REPORT_DRIVE = STAGE_ROOT / 'report'
MIN_DRIVE_FREE_BYTES = 10_000_000_000
subprocess.run(['df', '-h', '/content', '/content/drive/MyDrive'], check=True)
if shutil.disk_usage(DRIVE).free < MIN_DRIVE_FREE_BYTES:
    raise RuntimeError(f'Need at least {MIN_DRIVE_FREE_BYTES / 1e9:.0f} GB free on Drive for raw shards and reports.')
for path in (STAGE_ROOT, RAW_SCORE_DRIVE, CONFIG_DRIVE, REPORT_DRIVE):
    path.mkdir(parents=True, exist_ok=True)
print({'drive': str(DRIVE), 'stage_root': str(STAGE_ROOT)})

## 2. Clone, Pin, and Install

**One-time setup.** This run is pinned to exact producer and analysis revisions. Later report-only fixes may advance `ANALYSIS_SHA` without invalidating producer shards.

In [ ]:
# PYTHON CELL
import re

OLMO_REPO = 'https://github.com/myazdani/color-filter-olmo.git'
PRODUCER_SHA = 'a7a0bd55313255760c1e946f3e05a7b3767a858e'
ANALYSIS_SHA = 'cc8909a9062d6790e3c59dba445e39763add6a63'
NOTEBOOK_REVISION = 'broad-rate-sweep-v5-2026-07-17'
PRODUCER_DIR = Path('/content/color-filter-olmo-producer')
ANALYSIS_DIR = Path('/content/color-filter-olmo-analysis')
OLMO_DIR = PRODUCER_DIR
if not all(re.fullmatch(r'[0-9a-f]{40}', sha) for sha in (PRODUCER_SHA, ANALYSIS_SHA)):
    raise RuntimeError('Set both SHAs to full pushed commits before running Colab GPU work.')
def checkout_exact(repo_dir, sha):
    if not (repo_dir / '.git').is_dir():
        subprocess.run(['git', 'clone', OLMO_REPO, str(repo_dir)], check=True)
    subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', sha], check=True)
    subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', sha], check=True)
    actual = subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip()
    if actual != sha: raise RuntimeError(f'Commit mismatch: expected {sha}, found {actual}')
checkout_exact(PRODUCER_DIR, PRODUCER_SHA)
checkout_exact(ANALYSIS_DIR, ANALYSIS_SHA)
print({'producer_sha': PRODUCER_SHA, 'analysis_sha': ANALYSIS_SHA, 'notebook': NOTEBOOK_REVISION})

In [ ]:
# PYTHON CELL
from packaging.version import Version
import torch as colab_torch

torch_version = Version(colab_torch.__version__.split('+', 1)[0])
if sys.version_info[:2] != (3, 12) or not (Version('2.10') <= torch_version < Version('2.12')):
    raise RuntimeError(f'Unsupported runtime: Python {sys.version_info[:2]}, Torch {colab_torch.__version__}')
overlay = [
    'omegaconf==2.3.0', 'rich==13.9.4', 'cached_path==1.8.10', 'packaging==24.2',
    'boto3==1.35.94', 'google-cloud-storage==2.19.0', 'wandb==0.19.1',
    'torchmetrics==1.6.1', 'datasets==3.2.0', 'huggingface_hub==0.27.1',
    'transformers==4.47.1', 'tokenizers==0.21.0', 'pytest==8.3.4',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *overlay], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '-e', str(PRODUCER_DIR)], check=True)
print('installed pinned producer checkout without replacing Colab Torch')

In [ ]:
# PYTHON CELL
import importlib.util, numpy as np, pandas as pd, pyarrow, torch

sys.path.insert(0, str(PRODUCER_DIR))
metrics_source = (ANALYSIS_DIR / 'scripts/21_dropout_uncertainty_metrics.py').read_text()
required_markers = {
    'fixed score-index alignment': 'metadata_and_full_scores_indexed_by_score_index' in metrics_source,
    'bounded aggregation': '--max-rows' in metrics_source,
    'per-config K': 'config_num_samples' in (PRODUCER_DIR / 'scripts/targeted_dropout_colab_helpers.py').read_text(),
}
missing = [name for name, present in required_markers.items() if not present]
if missing:
    raise RuntimeError('Pinned revision lacks required capabilities: ' + ', '.join(missing))
spec = importlib.util.spec_from_file_location('broad_sweep_analysis', ANALYSIS_DIR / 'scripts/dropout_uncertainty_broad_sweep_colab.py')
if spec is None or spec.loader is None: raise RuntimeError('Could not load pinned analysis helper')
broad_helpers = importlib.util.module_from_spec(spec); sys.modules[spec.name] = broad_helpers; spec.loader.exec_module(broad_helpers)
if ANALYSIS_DIR.resolve() not in Path(broad_helpers.__file__).resolve().parents: raise RuntimeError('Broad helper is not loaded from ANALYSIS_SHA')
print('analysis helper:', Path(broad_helpers.__file__).resolve())
print('runtime:', {'torch': torch.__version__, 'cuda': torch.version.cuda, 'numpy': np.__version__})
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable after installation.')
gpu = torch.cuda.get_device_properties(0)
RUNTIME_IDENTITY = {
    'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.version.cuda,
    'gpu_name': torch.cuda.get_device_name(0), 'gpu_total_memory_bytes': int(gpu.total_memory),
}
print('runtime identity:', RUNTIME_IDENTITY)

## 3. Configure Paths and Sweep

All four configurations use broad dropout. Configuration IDs encode the decimal rate and remain isolated from the legacy `dropout_k8_p005` run.

In [ ]:
# PYTHON CELL
SWEEP_CONFIGS = [
    {'config_id': 'dropout_broad_k1_p000', 'dropout_rate': 0.0, 'num_samples': 1},
    {'config_id': 'dropout_broad_k8_p000001', 'dropout_rate': 0.00001, 'num_samples': 8},
    {'config_id': 'dropout_broad_k8_p0001', 'dropout_rate': 0.001, 'num_samples': 8},
    {'config_id': 'dropout_broad_k8_p001', 'dropout_rate': 0.01, 'num_samples': 8},
]
for config in SWEEP_CONFIGS:
    config.update({
        'dropout_target': 'attention+residual+embedding',
        'attention_dropout': config['dropout_rate'],
        'residual_dropout': config['dropout_rate'],
        'embedding_dropout': config['dropout_rate'],
    })
broad_helpers.validate_sweep_configs(SWEEP_CONFIGS)
print(pd.DataFrame(SWEEP_CONFIGS))

In [ ]:
# PYTHON CELL
ROWS, SEQ_LEN, GLOBAL_BATCH_SIZE = 500_000, 512, 32
SHARD_ROWS, SMOKE_ROWS, BENCH_ROWS = 24_992, 320, 1_024
INITIAL_MICROBATCH, SEED, FILE_SEQS = 16, 1, 1_048_576
TAU64_CUTOFF = 0.3513622284
TOKENS_DRIVE = DATA_DRIVE / 'score_pool_tokens_official_500k.npy'
META_DRIVE = DATA_DRIVE / 'score_pool_meta_official_500k.parquet'
FULL_SCORES_DRIVE = SOURCE_RESULTS / 'score-pool-robustness-official-500k' / 'scores_full.parquet'
PRIOR_CHECKPOINT = MODELS_DRIVE / 'prior'
BOOKS_CHECKPOINT = MODELS_DRIVE / 'conditional_books'
REFERENCE_P005 = SOURCE_RESULTS / 'dropout-uncertainty' / 'dropout_k8_p005' / 'alignment_fixed' / 'analysis'
LOCAL_WORK = Path('/content/dropout_uncertainty_broad_rate_sweep')
RUNTIME_CONFIG_DIR = LOCAL_WORK / 'runtime_configs'
RUNTIME_CHECKPOINT_DIR = LOCAL_WORK / 'runtime_checkpoints'
RUN_STATE_PATH = STAGE_ROOT / 'run_state.json'
SHARD_PLAN_PATH = STAGE_ROOT / 'shard_plan.json'
SUBSET_MANIFEST = STAGE_ROOT / 'subset_manifest.json'
SOURCE_ROWS = STAGE_ROOT / 'subset_source_rows.npy'
for path in (LOCAL_WORK, RUNTIME_CONFIG_DIR, RUNTIME_CHECKPOINT_DIR):
    path.mkdir(parents=True, exist_ok=True)

## 4. Validate Inputs and Prepare the Full Pool

**Safe to rerun.** This validates all source tables and checkpoints, then creates a runtime-local headerless `uint32` token file. The source identity and exact all-row mapping are fingerprinted on Drive.

In [ ]:
# PYTHON CELL
from scripts.targeted_dropout_colab_helpers import SubsetContext, prepare_fixed_subset

PREPARED = prepare_fixed_subset(SubsetContext(
    run_stage='stage_c_500k', subset_id='broad_rate_sweep_500k', stage_rows=ROWS,
    seq_len=SEQ_LEN, selection_seed=1729, expected_source_rows=ROWS, rows_per_stage_b_pool=20_000,
    tokens_path=TOKENS_DRIVE, metadata_path=META_DRIVE, full_scores_path=FULL_SCORES_DRIVE,
    prior_checkpoint=PRIOR_CHECKPOINT, books_checkpoint=BOOKS_CHECKPOINT, local_work=LOCAL_WORK,
    source_rows_path=SOURCE_ROWS, subset_manifest_path=SUBSET_MANIFEST,
    producer_sha=PRODUCER_SHA, analysis_sha=ANALYSIS_SHA, notebook_revision=NOTEBOOK_REVISION,
    runtime_identity=RUNTIME_IDENTITY,
))
subset_raw, subset_meta, subset_full = PREPARED.raw_tokens_path, PREPARED.metadata_path, PREPARED.full_scores_path
print({'raw_GB': round(subset_raw.stat().st_size / 1e9, 3), 'pools': PREPARED.pool_counts})

In [ ]:
# PYTHON CELL
from scripts.targeted_dropout_colab_helpers import build_shard_plan
SHARDS = build_shard_plan(ROWS, SHARD_ROWS, GLOBAL_BATCH_SIZE, SHARD_PLAN_PATH)
print({'shards': len(SHARDS), 'first': SHARDS[0], 'last': SHARDS[-1]})

## 5. Scoring Workflow

The notebook delegates scoring, resume validation, analysis, reporting, and bundling to pushed helper modules.

In [ ]:
# PYTHON CELL
from scripts.targeted_dropout_colab_helpers import ScoringContext, TargetedDropoutRunner
BroadDropoutSweep, BroadSweepContext = broad_helpers.BroadDropoutSweep, broad_helpers.BroadSweepContext

os.environ['PYTHONUNBUFFERED'] = '1'
os.environ.setdefault('WANDB_MODE', 'disabled')
RUNNER = TargetedDropoutRunner(ScoringContext(
    olmo_dir=OLMO_DIR, template_config=OLMO_DIR / 'configs/sweeps/score-parallel-dropout-uncertainty.yaml',
    runtime_checkpoint_dir=RUNTIME_CHECKPOINT_DIR, runtime_config_dir=RUNTIME_CONFIG_DIR,
    config_drive=CONFIG_DRIVE, raw_score_drive=RAW_SCORE_DRIVE, stage_root=STAGE_ROOT,
    subset_raw=subset_raw, run_state_path=RUN_STATE_PATH, producer_sha=PRODUCER_SHA,
    analysis_sha=ANALYSIS_SHA, notebook_revision=NOTEBOOK_REVISION, run_stage='broad_rate_sweep_500k',
    subset_id='broad_rate_sweep_500k', subset_fingerprint=PREPARED.subset_fingerprint,
    runtime_identity=RUNTIME_IDENTITY, checkpoint_identities=PREPARED.checkpoint_identities,
    seed=SEED, num_samples=8, global_batch_size=GLOBAL_BATCH_SIZE, stage_rows=ROWS,
    shard_rows=SHARD_ROWS, file_seqs=FILE_SEQS,
))

In [ ]:
# PYTHON CELL
WORKFLOW = BroadDropoutSweep(BroadSweepContext(
    runner=RUNNER, analysis_olmo_dir=ANALYSIS_DIR, configs=SWEEP_CONFIGS, shards=SHARDS, prior_checkpoint=PRIOR_CHECKPOINT,
    books_checkpoint=BOOKS_CHECKPOINT, metadata_path=subset_meta, full_scores_path=subset_full,
    stage_root=STAGE_ROOT, report_dir=REPORT_DRIVE, config_drive=CONFIG_DRIVE,
    run_state_path=RUN_STATE_PATH, shard_plan_path=SHARD_PLAN_PATH, subset_manifest_path=SUBSET_MANIFEST,
    producer_sha=PRODUCER_SHA, analysis_sha=ANALYSIS_SHA, notebook_revision=NOTEBOOK_REVISION,
    rows=ROWS, seq_len=SEQ_LEN, tau64_cutoff=TAU64_CUTOFF, global_batch_size=GLOBAL_BATCH_SIZE,
    shard_rows=SHARD_ROWS, microbatch=INITIAL_MICROBATCH,
    reference_p005_analysis=REFERENCE_P005,
))
probe = RUNNER.build_score_config(
    SWEEP_CONFIGS[0], 'prior_probe', PRIOR_CHECKPOINT, STAGE_ROOT / '_config_probe',
    {'start': 0, 'end': 32, 'rows': 32, 'data_start_step': 0}, INITIAL_MICROBATCH,
)
assert probe.uncertainty_scoring.num_samples == 1
print('per-config K and runtime config probe passed')

## 6. Cheap Smoke Test / GPU Gate

**Safe to rerun.** This scores exactly 320 rows for both models at `p=0`, verifies corrected alignment, and requires strong agreement with deterministic full scores.

In [ ]:
# PYTHON CELL
gate = ANALYSIS_DIR / 'tests/dropout_uncertainty_broad_sweep_colab_test.py'
subprocess.run(
    [sys.executable, '-m', 'pytest', '--confcutdir=tests', '-q', str(gate)],
    cwd=str(ANALYSIS_DIR), check=True,
)
print('synthetic broad-sweep gate passed')

In [ ]:
# PYTHON CELL
smoke = WORKFLOW.run_zero_dropout_smoke(SMOKE_ROWS)
print('zero-dropout smoke passed:', smoke)

## 7. Batch and Shard Tuning

**Benchmark only. Safe to rerun.** This scores exactly 640 rows for the `p=0.01`, `K=8` prior model at each candidate microbatch. It persists the selected microbatch and prints a conservative production ETA.

In [ ]:
# PYTHON CELL
run_state = WORKFLOW.benchmark(BENCH_ROWS, candidates=[(32, 16), (64, 32), (128, 64), (256, 128), (512, 256)])
print(json.dumps(run_state, indent=2, sort_keys=True))

## 8. Full Resumable Sweep

**Full run. Safe to rerun.** Fingerprint-matched shards are skipped. Before every shard the cell prints the configuration, model, row range, action, and updated ETA; scorer output is streamed live and copied to its Drive log.

In [ ]:
# PYTHON CELL
RUN_FULL_SWEEP = True
if not RUN_FULL_SWEEP:
    raise RuntimeError('Set RUN_FULL_SWEEP=True after reviewing Sections 0-7.')
WORKFLOW.run_production()

## 9. Resume After Disconnect / Status

**Safe to rerun.** After reconnecting, rerun Sections 0-5, skip smoke and benchmark when `run_state.json` is valid, then run this bounded status cell. Rerun Section 8 to fill only missing or invalid shards.

In [ ]:
# PYTHON CELL
status = pd.DataFrame(WORKFLOW.raw_status())
print(status.groupby(['config_id', 'model_id'])['valid'].agg(['sum', 'count']))
missing = status.loc[~status['valid']]
print('missing/invalid jobs:', len(missing))
if len(missing):
    print(missing.head(10).to_string(index=False))
else:
    print('raw scoring grid is complete')

## 10. Corrected Metrics and Report

**Safe to rerun after raw completion.** Analysis uses explicit row limits and the fixed `score_index` alignment contract. Complete fingerprint-matched analysis is reused; stale isolated analysis is regenerated.

In [ ]:
# PYTHON CELL
WORKFLOW.analyze()
report_result = WORKFLOW.build_report()
print(report_result['summary'].to_string(index=False))
print('acceptance:', report_result['acceptance'])
print('report:', REPORT_DRIVE / 'report.md')

## 11. Output Review, Bundle, and Download

Review the full-pool row counts, corrected alignment contract, zero-dropout acceptance, per-task metrics, and runtime telemetry before packaging.

In [ ]:
# PYTHON CELL
acceptance = json.loads((REPORT_DRIVE / 'sweep_acceptance.json').read_text())
summary = pd.read_csv(REPORT_DRIVE / 'sweep_summary.csv')
if not acceptance['all_configs_complete'] or not acceptance['zero_dropout_control_passed']:
    raise RuntimeError(f'Sweep acceptance failed: {acceptance}')
if int((summary['source'] == 'sweep').sum()) != len(SWEEP_CONFIGS):
    raise RuntimeError('Sweep summary does not contain exactly four new configurations.')
print(summary.to_string(index=False))
print('output review passed')

In [ ]:
# PYTHON CELL
LOCAL_ARCHIVE = Path('/content/dropout_uncertainty_broad_rate_sweep_bundle.zip')
DRIVE_ARCHIVE = STAGE_ROOT / 'dropout_uncertainty_broad_rate_sweep_bundle.zip'
bundle = WORKFLOW.build_bundle(LOCAL_ARCHIVE, DRIVE_ARCHIVE)
print({'drive_archive': str(bundle['drive_archive']), 'file_count': bundle['file_count']})

In [ ]:
# DOWNLOAD-ONLY CELL
from google.colab import files
from scripts.targeted_dropout_colab_helpers import verify_bundle_archive

AUTO_DOWNLOAD = globals().get('AUTO_DOWNLOAD', True)
members = verify_bundle_archive(DRIVE_ARCHIVE)
print('verified bundle members:', len(members))
if AUTO_DOWNLOAD:
    files.download(str(DRIVE_ARCHIVE))
print('Drive fallback:', DRIVE_ARCHIVE)